# Prediksi Tingkat Obesitas dari Kebiasaan Makan dan Gaya Hidup
**Kelompok 5 — STMIK Pontianak — Predictive Analytics**

Pertanyaan: dari variabel kebiasaan makan dan gaya hidup sehari-hari, apakah seorang responden masuk kelompok **Kurang, Normal, atau Lebih**? Lebih berarti berat badan berlebih.

Datanya: Obesity Prediction Dataset (Kaggle, ruchikakumbhar), 2.111 responden dari Meksiko, Peru, dan Kolombia, jadi bukan data Indonesia, dan hasilnya dibaca sebagai pola kebiasaan, bukan potret orang Indonesia.

Jalankan sel demi sel dari atas ke bawah (Runtime > Run all di Colab).

Peta analisis: Bagian 1 analisis deskriptif, Bagian 2–6 analisis eksplorasi data — univariat, bivariat, multivariat —, Bagian 7–9, 11, dan 12 analisis prediktif, Bagian 10 analisis mekanistik, dan Bagian 13 analisis preskriptif.


## 1. Muat data — Analisis Deskriptif
Sumber CSV: file `kelompok-5-dataset-obesitas-2111.csv` di repo yang sama. Di Colab, sel di bawah baca langsung dari GitHub. Dataset ini punya 17 atribut, dan sel di bawah memuatnya jadi DataFrame.

Bagian ini analisis deskriptif: gambaran datasetnya, siapa respondennya, dan atribut apa saja yang dicatat.


In [1]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/awangsuryarmdhn/weight-loss-regression/main/kelompok-5-dataset-obesitas-2111.csv"
df = pd.read_csv(URL)
df.columns = [c.strip() for c in df.columns]
print(df.shape)
print(df.columns.tolist())
print("Data kosong:", int(df.isna().sum().sum()))
df.head()

(2111, 17)
['Gender', 'Age', 'Height', 'Weight', 'family_history', 'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', 'Obesity']
Data kosong: 0


,Gender,Age,Height,Weight,family_history,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,Obesity
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


## 2. Kenali targetnya: 7 level — Analisis Univariat
Target aslinya ada 7 level, dan isi tiap level hampir sama banyak, 12,9% sampai 16,6%, jadi tidak ada level yang mendominasi, dan model tidak bisa menang cuma dengan menebak level terbanyak.

Ini analisis univariat: satu variabel sekali, tujuannya tahu bentuk distribusi target sebelum melihat hubungannya dengan variabel lain.


In [2]:
print(df["Obesity"].value_counts())
print(df["Obesity"].value_counts(normalize=True).round(3))
print("\nBaseline 7 kelas (selalu tebak level terbanyak):",
      round(df["Obesity"].value_counts(normalize=True).max() * 100, 2), "%")

Obesity
Obesity_Type_I         351
Obesity_Type_III       324
Obesity_Type_II        297
Overweight_Level_I     290
Overweight_Level_II    290
Normal_Weight          287
Insufficient_Weight    272
Name: count, dtype: int64
Obesity
Obesity_Type_I         0.166
Obesity_Type_III       0.153
Obesity_Type_II        0.141
Overweight_Level_I     0.137
Overweight_Level_II    0.137
Normal_Weight          0.136
Insufficient_Weight    0.129
Name: proportion, dtype: float64

Baseline 7 kelas (selalu tebak level terbanyak): 16.63 %


## 3. Sederhanakan 7 level menjadi 3 kelas — Persiapan EDA
7 level terlalu rinci buat presentasi, karena beda Overweight I dan II, atau Obesitas I dan II, itu soal derajat, bukan soal lebih atau normal, jadi ketujuhnya digabung jadi 3 kelas: **Kurang, Normal, Lebih**. Kelas Lebih jadi besar, 73,5%, jadi baseline-nya ikut naik, dan ini penting diingat waktu baca hasil model, karena akurasi harus dibaca bareng baseline.


In [3]:
def ke_3_kelas(x):
    if x == "Insufficient_Weight":
        return "Kurang"
    if x == "Normal_Weight":
        return "Normal"
    return "Lebih"

df["Kelas3"] = df["Obesity"].apply(ke_3_kelas)
print(df["Kelas3"].value_counts())
print(df["Kelas3"].value_counts(normalize=True).round(3))
print("\nBaseline 3 kelas (selalu tebak 'Lebih'):", round(df["Kelas3"].value_counts(normalize=True).max() * 100, 2), "%")

Kelas3
Lebih     1552
Normal     287
Kurang     272
Name: count, dtype: int64
Kelas3
Lebih     0.735
Normal    0.136
Kurang    0.129
Name: proportion, dtype: float64

Baseline 3 kelas (selalu tebak 'Lebih'): 73.52 %


## 4. Analisis Univariat: gambaran responden dan variabel kebiasaan
Analisis univariat melihat satu variabel sekali, tujuannya tahu bentuk datanya: siapa respondennya, bagaimana sebaran tiap atribut, dan apakah ada yang janggal sebelum lanjut ke hubungan antar variabel.


In [4]:
print("Jenis kelamin:")
print(df["Gender"].value_counts())
print(df["Gender"].value_counts(normalize=True).round(3))
print("\nUmur:")
print(df["Age"].describe().round(2))
print("\nRiwayat keluarga overweight (ya):", round((df["family_history"] == "yes").mean(), 3))
print("Sering makan tinggi kalori (FAVC = ya):", round((df["FAVC"] == "yes").mean(), 3))
print("\nNgemil (CAEC):")
print(df["CAEC"].value_counts(normalize=True).round(3))
print("\nMerokok (ya):", round((df["SMOKE"] == "yes").mean(), 3))
print("Memantau kalori (ya):", round((df["SCC"] == "yes").mean(), 3))
print("\nAlkohol (CALC):")
print(df["CALC"].value_counts(normalize=True).round(3))
print("\nTransportasi (MTRANS):")
print(df["MTRANS"].value_counts(normalize=True).round(3))
print("\nRata-rata variabel numerik:")
print(df[["FCVC", "NCP", "CH2O", "FAF", "TUE"]].mean().round(2))


Jenis kelamin:
Gender
Male      1068
Female    1043
Name: count, dtype: int64
Gender
Male      0.506
Female    0.494
Name: proportion, dtype: float64

Umur:
count    2111.00
mean       24.31
std         6.35
min        14.00
25%        19.95
50%        22.78
75%        26.00
max        61.00
Name: Age, dtype: float64

Riwayat keluarga overweight (ya): 0.818
Sering makan tinggi kalori (FAVC = ya): 0.884

Ngemil (CAEC):
CAEC
Sometimes     0.836
Frequently    0.115
Always        0.025
no            0.024
Name: proportion, dtype: float64

Merokok (ya): 0.021
Memantau kalori (ya): 0.045

Alkohol (CALC):
CALC
Sometimes     0.664
no            0.303
Frequently    0.033
Always        0.000
Name: proportion, dtype: float64

Transportasi (MTRANS):
MTRANS
Public_Transportation    0.748
Automobile               0.216
Walking                  0.027
Motorbike                0.005
Bike                     0.003
Name: proportion, dtype: float64

Rata-rata variabel numerik:
FCVC    2.42
NCP     2.69
CH

## 5. Analisis Bivariat: kebiasaan vs kelas
Ini analisis bivariat: satu variabel kebiasaan lawan tingkat obesitas. Sebelum membuat model, lihat dulu pola kasarnya: kelompok Lebih lebih tua, lebih jarang olahraga, dan jauh lebih sering punya riwayat keluarga overweight, jadi tiga variabel ini yang pertama kali kelihatan membedakan kelompok.


In [5]:
print(df.groupby("Kelas3")[["Age", "FCVC", "NCP", "CH2O", "FAF", "TUE"]].mean().round(2))
print("\nSering makan tinggi kalori (FAVC = ya):")
print(df.groupby("Kelas3")["FAVC"].apply(lambda s: (s == "yes").mean()).round(3))
print("\nPunya riwayat keluarga overweight:")
print(df.groupby("Kelas3")["family_history"].apply(lambda s: (s == "yes").mean()).round(3))
print("\nNgemil kadang-kadang (CAEC = Sometimes):")
print(df.groupby("Kelas3")["CAEC"].apply(lambda s: (s == "Sometimes").mean()).round(3))

          Age  FCVC   NCP  CH2O   FAF   TUE
Kelas3                                     
Kurang  19.78  2.48  2.91  1.87  1.25  0.84
Lebih   25.58  2.42  2.64  2.06  0.92  0.62
Normal  21.74  2.33  2.74  1.85  1.25  0.68

Sering makan tinggi kalori (FAVC = ya):
Kelas3
Kurang    0.812
Lebih     0.926
Normal    0.725
Name: FAVC, dtype: float64

Punya riwayat keluarga overweight:
Kelas3
Kurang    0.463
Lebih     0.931
Normal    0.540
Name: family_history, dtype: float64

Ngemil kadang-kadang (CAEC = Sometimes):
Kelas3
Kurang    0.537
Lebih     0.941
Normal    0.554
Name: CAEC, dtype: float64


## 6. Analisis Multivariat: korelasi antar variabel
Analisis multivariat melihat banyak variabel sekaligus, tujuannya tahu apakah tiap variabel membawa informasi sendiri atau cuma mengulang informasi variabel lain, dan ini dibaca dari korelasi antar variabel numerik.


In [6]:
# Level obesitas diberi skor 0-6 (Kurang -> Obesitas III) supaya korelasinya bisa dihitung.
# Skor ini cuma dipakai di sel ini, tidak ditambahkan ke DataFrame, supaya tidak bocor ke model.
skor = {"Insufficient_Weight": 0, "Normal_Weight": 1, "Overweight_Level_I": 2,
        "Overweight_Level_II": 3, "Obesity_Type_I": 4, "Obesity_Type_II": 5, "Obesity_Type_III": 6}
skor_level = df["Obesity"].map(skor)
angka = ["Age", "FCVC", "NCP", "CH2O", "FAF", "TUE"]
print("Korelasi Pearson tiap variabel numerik dengan level obesitas (skor 0-6):")
print(df[angka].corrwith(skor_level).round(2))
print("\nKorelasi antar variabel numerik:")
print(df[angka].corr().round(2))


Korelasi Pearson tiap variabel numerik dengan level obesitas (skor 0-6):
Age     0.28
FCVC    0.23
NCP     0.03
CH2O    0.13
FAF    -0.20
TUE    -0.11
dtype: float64

Korelasi antar variabel numerik:
       Age  FCVC   NCP  CH2O   FAF   TUE
Age   1.00  0.02 -0.04 -0.05 -0.14 -0.30
FCVC  0.02  1.00  0.04  0.07  0.02 -0.10
NCP  -0.04  0.04  1.00  0.06  0.13  0.04
CH2O -0.05  0.07  0.06  1.00  0.17  0.01
FAF  -0.14  0.02  0.13  0.17  1.00  0.06
TUE  -0.30 -0.10  0.04  0.01  0.06  1.00


Korelasi tiap variabel dengan level obesitas semuanya lemah sampai sedang, paling besar umur 0,28, jadi tidak ada satu variabel numerik yang sendirian menjelaskan tingkat obesitas. Antar variabel kebiasaan korelasinya juga lemah, yang paling kuat umur lawan waktu layar -0,30, jadi tiap variabel sebagian besar membawa informasi sendiri-sendiri, dan ini cocok dengan Bagian 11, di mana delapan variabel teratas dipakai model totalnya 78,3% dan tidak ada satu variabel yang mendominasi.


## 7. Siapkan bahan model: hanya kebiasaan — Analisis Prediktif
Tinggi dan berat badan ada di data, tapi tidak dipakai sebagai prediktor, karena model hanya membaca variabel kebiasaan: umur, riwayat keluarga, pola makan, olahraga, waktu layar, dan transportasi.

Tiga model kami bandingkan: Decision Tree (paling dalam 6 tingkat, supaya aturannya masih bisa dijelaskan), Random Forest (200 pohon), dan Regresi Logistik. Data dibagi 80% buat latih, 20% buat uji, diacak pakai seed 42 supaya hasilnya bisa diulang.


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

X_kebiasaan = df.drop(columns=["Obesity", "Kelas3", "Height", "Weight"])

def latih(X, y, label):
    X_latih, X_uji, y_latih, y_uji = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)
    angka = X.select_dtypes(include=[np.number]).columns.tolist()
    kategori = [c for c in X.columns if c not in angka]
    pra = ColumnTransformer([
        ("angka", StandardScaler(), angka),
        ("kategori", OneHotEncoder(handle_unknown="ignore"), kategori)])
    hasil = {}
    for nama, model in [
        ("Decision Tree", DecisionTreeClassifier(random_state=42, max_depth=6)),
        ("Random Forest", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
        ("Regresi Logistik", LogisticRegression(max_iter=1000)),
    ]:
        pipa = Pipeline([("pra", pra), ("model", model)])
        pipa.fit(X_latih, y_latih)
        akurasi = accuracy_score(y_uji, pipa.predict(X_uji))
        hasil[nama] = round(float(akurasi), 4)
        print(f"{label} — {nama}: {akurasi:.2%}")
    return hasil

## 8. Model 7 level, dari kebiasaan saja — Analisis Prediktif
Menebak 1 dari 7 level yang saling berdekatan memang sulit, karena batas antar level tipis, tapi Random Forest akurasinya 85,34%, jauh di atas baseline 16,63%, jadi model ini memang menangkap pola kebiasaannya, bukan cuma menebak.


In [8]:
latih(X_kebiasaan, df["Obesity"], "7 kelas, kebiasaan saja")

7 kelas, kebiasaan saja — Decision Tree: 66.43%


7 kelas, kebiasaan saja — Random Forest: 85.34%


7 kelas, kebiasaan saja — Regresi Logistik: 62.41%


{'Decision Tree': 0.6643, 'Random Forest': 0.8534, 'Regresi Logistik': 0.6241}

## 9. Model 3 kelas, dari kebiasaan saja: MODEL UTAMA — Analisis Prediktif
Ini model utama yang dipakai. Random Forest akurasinya 90,31%, jelas di atas baseline 73,52%, karena model ini menggabungkan banyak pohon keputusan sehingga kesalahannya saling menutupi. Decision Tree dan Regresi Logistik hampir sama, 85,34% dan 85,11%, jadi hasilnya tidak cuma datang dari satu model.


In [9]:
hasil_utama = latih(X_kebiasaan, df["Kelas3"], "3 kelas, kebiasaan saja (MODEL UTAMA)")
hasil_utama

3 kelas, kebiasaan saja (MODEL UTAMA) — Decision Tree: 85.34%


3 kelas, kebiasaan saja (MODEL UTAMA) — Random Forest: 90.31%
3 kelas, kebiasaan saja (MODEL UTAMA) — Regresi Logistik: 85.11%


{'Decision Tree': 0.8534, 'Random Forest': 0.9031, 'Regresi Logistik': 0.8511}

## 10. Analisis Mekanistik: bagaimana model dan kebiasaan bekerja

Bagian ini analisis mekanistik: bagaimana sesuatu bekerja, bukan cuma apa hasilnya.

**Mekanisme model.** Random Forest membuat banyak pohon keputusan, dan tiap pohon memilih satu kelas. Kelas yang paling banyak dipilih pohon itulah jawaban model, jadi kalau satu pohon salah, pohon lain bisa menutupi, dan hasilnya lebih stabil daripada satu pohon saja. Tiap pohon dibuat dari data latih yang diacak dan cuma melihat sebagian atribut waktu bercabang, jadi pohon-pohonnya berbeda satu sama lain. Inilah kenapa akurasi Random Forest 90,31% buat 3 kelas, paling tinggi dari tiga model.

**Mekanisme kebiasaan.** Ngemil dan makanan tinggi kalori bekerja lewat asupan energi: ngemil menambah makan di luar jam makan utama, dan makanan tinggi kalori menambah energi per porsinya. Olahraga dan jalan kaki bekerja sebaliknya, memakai energi. Riwayat keluarga bekerja lewat pola makan yang dibiasakan di keluarga dan faktor bawaan tubuh seperti metabolisme. Umur bekerja lewat rutinitas yang berubah seiring waktu, dan dibaca sebagai konteks, bukan sebab.

**Batasnya:** penjelasan mekanisme di atas itu cara kerja yang masuk akal dari pola di data, bukan bukti sebab-akibat dari analisis ini, karena datanya survei satu waktu dan yang kelihatan cuma kaitan antar variabel.


## 11. Variabel apa yang paling berpengaruh? — Analisis Prediktif (Multivariat)
Tidak ada satu variabel yang mendominasi. Angka di bawah sudah digabung per variabel, karena model awalnya memisahkan jawaban ya dan tidak ke baris yang berbeda, jadi kalau tidak digabung, bobot satu variabel kelihatan terpecah.


In [10]:
from collections import defaultdict

X_latih, X_uji, y_latih, y_uji = train_test_split(
    X_kebiasaan, df["Kelas3"], test_size=0.2, random_state=42, stratify=df["Kelas3"])
angka = X_kebiasaan.select_dtypes(include=[np.number]).columns.tolist()
kategori = [c for c in X_kebiasaan.columns if c not in angka]
pra = ColumnTransformer([
    ("angka", "passthrough", angka),
    ("kategori", OneHotEncoder(handle_unknown="ignore"), kategori)])
rf = Pipeline([("pra", pra), ("model", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))])
rf.fit(X_latih, y_latih)
nama_fitur = pra.fit(X_latih).get_feature_names_out()
penting = sorted(zip(nama_fitur, rf.named_steps["model"].feature_importances_), key=lambda t: -t[1])

gabung = defaultdict(float)
for nama, nilai in penting:
    dasar = nama.split("__", 1)[1]
    for awalan in ["family_history", "CAEC", "Gender", "MTRANS", "CALC", "SCC", "FAVC", "SMOKE"]:
        if dasar.startswith(awalan + "_"):
            dasar = awalan
            break
    gabung[dasar] += nilai
for nama, nilai in sorted(gabung.items(), key=lambda t: -t[1]):
    print(f"{nama}: {nilai:.3f}")

CAEC: 0.156
Age: 0.137
family_history: 0.134
NCP: 0.080
FAF: 0.078
CH2O: 0.069
TUE: 0.068
FCVC: 0.061
MTRANS: 0.057
Gender: 0.051
CALC: 0.050
FAVC: 0.030
SCC: 0.023
SMOKE: 0.007


## 12. Demo: dua profil kebiasaan — Analisis Prediktif
Profil sehat: 22 tahun, tidak ada riwayat keluarga overweight, jarang makanan tinggi kalori, sayur tiap makan, olahraga teratur, jalan kaki, jadi hampir semua variabel kebiasaannya mengarah ke Normal.
Profil berisiko: 28 tahun, ada riwayat keluarga, sering makanan tinggi kalori, jarang sayur, jarang olahraga, sering ngemil, banyak waktu layar, jadi variabelnya kebanyakan mengarah ke Lebih.
Perhatikan: model memberi peluang, bukan vonis, dan profil berisiko pun masih punya peluang Normal yang cukup besar, 40,5%, jadi modelnya tidak yakin penuh.


In [11]:
profil_sehat = {
    "Gender": "Female", "Age": 22, "family_history": "no", "FAVC": "no",
    "FCVC": 3.0, "NCP": 3.0, "CAEC": "no", "SMOKE": "no", "CH2O": 2.0,
    "SCC": "no", "FAF": 2.0, "TUE": 1.0, "CALC": "no", "MTRANS": "Walking"}
profil_berisiko = {
    "Gender": "Male", "Age": 28, "family_history": "yes", "FAVC": "yes",
    "FCVC": 1.0, "NCP": 2.0, "CAEC": "Frequently", "SMOKE": "no", "CH2O": 1.0,
    "SCC": "no", "FAF": 0.0, "TUE": 3.0, "CALC": "Sometimes", "MTRANS": "Automobile"}

# Model final dilatih ulang pada SEMUA data (bukan hanya data latih 80%),
# supaya demo memakai model terbaik yang kita punya.
rf_final = Pipeline([("pra", pra), ("model", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))])
rf_final.fit(X_kebiasaan, df["Kelas3"])
rf = rf_final

for label, profil in [("Profil sehat", profil_sehat), ("Profil berisiko", profil_berisiko)]:
    baris = pd.DataFrame([profil])
    prediksi = rf.predict(baris)[0]
    peluang = dict(zip(rf.classes_, rf.predict_proba(baris)[0].round(3)))
    print(label, "->", prediksi, peluang)

Profil sehat -> Normal {'Kurang': 0.26, 'Lebih': 0.085, 'Normal': 0.655}
Profil berisiko -> Lebih {'Kurang': 0.045, 'Lebih': 0.55, 'Normal': 0.405}


## 13. Analisis Preskriptif: solusi berdasarkan model

Bagian ini analisis preskriptif: dari hasil model, ditentukan aksi apa yang dilakukan, buat siapa, dan kenapa aksi itu yang dipilih.

- **Kelompok Lebih (1.552 responden):** kurangi makanan tinggi kalori satu porsi sehari dan naikkan olahraga pelan-pelan, karena FAVC bedanya paling besar antar kelas (92,6% lawan 72,5% di kelompok Normal) dan skor olahraga kelompok ini paling rendah (0,92).
- **Yang punya riwayat keluarga overweight:** jadi prioritas skrining dini, karena 93,1% kelompok Lebih punya riwayat keluarga, tapi riwayat menaikkan risiko, bukan takdir.
- **Kampus dan kantin:** sayur jadi pilihan utama, air minum gratis dan gampang dijangkau, karena makan sayur rata-rata cuma 2,33–2,48 dari skala 3 dan air minum 1,85–2,06 liter sehari.
- **Mahasiswa, mulai minggu ini:** ganti 1 camilan tinggi kalori sehari dengan buah atau sayur, minum air 2 liter sehari, olahraga ringan 3 kali seminggu 30 menit, dan makan utama teratur 3 kali sehari.
- **Kelompok Kurang (272 responden):** jangan cuma fokus ke yang berat badan lebih, kelompok ini butuh makan cukup dan teratur, dan kalau berat badan terus turun, cek ke tenaga kesehatan.


## 14. Kesimpulan
1. Dari variabel kebiasaan saja, model bisa membedakan Kurang, Normal, dan Lebih, dan Random Forest akurasinya 90,31%, jelas di atas baseline 73,52%, karena model ini menggabungkan banyak pohon keputusan. Buat 7 level, akurasinya 85,34% lawan baseline 16,63%.
2. Tidak ada satu variabel yang menentukan semuanya. Pola ngemil (15,6%), umur (13,7%), dan riwayat keluarga (13,4%) paling banyak dipakai model, lalu jumlah makan, olahraga, air minum, dan waktu layar, masing-masing sekitar 7–8%, jadi model ini membaca banyak variabel sekaligus, bukan cuma satu.
3. Riwayat keluarga menaikkan risiko, tapi bukan takdir: 54,0% kelompok Normal juga punya riwayat keluarga overweight, jadi variabel ini menaikkan peluang, bukan menentukan hasil.
4. Ini prediksi risiko dari survei kebiasaan, bukan diagnosis medis, dan datanya dari Meksiko, Peru, dan Kolombia, bukan Indonesia, jadi hasilnya perlu diuji lagi di data Indonesia sebelum dipakai buat kebijakan.

Sumber data: https://www.kaggle.com/datasets/ruchikakumbhar/obesity-prediction
